In [1]:
import onnx
from optimum.onnxruntime import ORTQuantizer, ORTModelForSequenceClassification
from functools import partial
from transformers import AutoTokenizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig, AutoCalibrationConfig, QuantizationConfig
from onnxruntime.quantization import QuantFormat, QuantizationMode, QuantType
from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import AutoCalibrationConfig, QuantizationConfig
from optimum.onnxruntime.modeling_ort import ORTModelForImageClassification
from optimum.onnxruntime.preprocessors import QuantizationPreprocessor
from optimum.onnxruntime.preprocessors.passes import (
    ExcludeGeLUNodes,
    ExcludeLayerNormNodes,
    ExcludeNodeAfter,
    ExcludeNodeFollowedBy,
)
from optimum.onnxruntime.utils import evaluation_loop
from datasets import load_dataset
from torchvision.transforms import CenterCrop, Compose, Normalize, Resize, ToTensor

import  torch
import numpy as np
from ultralytics.data.augment import LetterBox
from _dataloader import Processor, transform, custom_collate_fn





In [3]:
quantizer = ORTQuantizer.from_pretrained('.')

Could not load the config for yolo.onnx automatically, this might make the quantized model harder to use because it will not be able to be loaded by an ORTModel without having to specify the configuration explicitly.


In [7]:
qconfig = QuantizationConfig(
is_static             = True,
format                = QuantFormat.QDQ,
mode                  = QuantizationMode.QLinearOps,
activations_dtype     = QuantType.QInt8,
weights_dtype         = QuantType.QInt8,
per_channel           = True,
reduce_range          = False,
operators_to_quantize = ["MatMul", "Conv", "Gemm"],
)

In [4]:
dataset = load_dataset(path='rafaelpadilla/coco2017', cache_dir='/Data/Dataset/COCO', split='val')
calibration_ds = dataset.with_transform(lambda batch: transform(batch, Processor())).shuffle(seed=42).select(range(2000))

Resolving data files:   0%|          | 0/39 [00:00<?, ?it/s]

In [5]:
calibration_config =  AutoCalibrationConfig.percentiles(calibration_ds)

In [8]:
ranges = quantizer.fit(
        dataset = calibration_ds,
        calibration_config=calibration_config,
        operators_to_quantize=qconfig.operators_to_quantize
)

RuntimeError: Failed to find input with name: image_id in the model input def list

In [3]:
model = onnx.load("yolov8s.onnx")

In [4]:
graph = model.graph

In [9]:
old_name = "images"
new_name = "image"

In [10]:
for inp in graph.input:
    if inp.name == old_name:
        inp.name = new_name

In [11]:
for init in graph.initializer:
    if init.name == old_name:
        init.name = new_name

In [12]:
for node in graph.node:
    for idx, name in enumerate(node.input):
        if name == old_name:
            node.input[idx] = new_name

In [13]:
onnx.checker.check_model(model)

In [14]:
onnx.save(model, "yolo.onnx")

In [16]:
onnx.checker.check_model(model)

In [17]:
model.graph.input

[name: "image"
type {
  tensor_type {
    elem_type: 1
    shape {
      dim {
        dim_param: "batch"
      }
      dim {
        dim_value: 3
      }
      dim {
        dim_param: "height"
      }
      dim {
        dim_param: "width"
      }
    }
  }
}
]

In [18]:
onnx.save(model, "yolo.onnx")